In [2]:
from selenium import webdriver
import time
import requests #fetches data from webpage
from bs4 import BeautifulSoup #extracts data we need
import pandas as pd #to store data as csv or excel format
from datetime import date
import re

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.dcbbank.com/rates/fixed-deposit-interest-rate" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Set up the Chrome driver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)
driver.get(URL) # Fetch the HTML page
time.sleep(5)  ## Wait for 5 secs the JavaScript to execute (you can adjust the sleep time if needed)
html = driver.page_source # Get the rendered HTML
driver.quit() # Close the driver

soup=BeautifulSoup(html,"html.parser")
#print(soup.prettify())
table=soup.find_all("table")
table=table[0]
rows=table.find_all("tr")

print("Today's date =",date1)
print("Interest rate file name = ",file_name)

list_of_interest_rates = []
tax_saver_row = []
row_count = 0
for i in rows[1:]:  #Skip heading
    data=i.find_all("td")
    row=[tr.text for tr in data]
    list_of_interest_rates.append(row)

final_rates=[]
tax_saver_row = []
row_count = 0
for i in list_of_interest_rates:
    x=['Ujjivan SF Bank']
    x.extend(i)
    #print(x)
    x.extend(['',str(float(x[2][0:4]) + 0.5) + '%',''])
    if row_count == 9:
        tax_saver_row = ['Ujjivan SF Bank', 'Tax-saver Fixed Deposit', x[2], x[3], x[4], x[5]]
        final_rates.append(tax_saver_row)
    final_rates.append(x)
    row_count += 1
#print(final_rates)

pd.set_option('display.max_columns', None)
int_rate_df=pd.DataFrame(final_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                        'sr_rate','annualised_sr_rate'])
#print(list_of_interest_rates)
#print(int_rate_df)
int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
print("\nUjjivan Bank = Success. Number of rows added = ", len(final_rates))

Today's date = 12012025
Interest rate file name =  c:\Interest_rate_files\Interest_rate_12012025.csv
          bank_name                                  tenure gen_rate  \
0   Ujjivan SF Bank                       7 days to 29 days    3.75%   
1   Ujjivan SF Bank                      30 days to 89 days    4.25%   
2   Ujjivan SF Bank                     90 days to 179 days    4.75%   
3   Ujjivan SF Bank                  6 months to < 9 months    7.00%   
4   Ujjivan SF Bank                                9 months    7.50%   
5   Ujjivan SF Bank            9 months 1 day to <12 months    7.00%   
6   Ujjivan SF Bank                               12 months    8.25%   
7   Ujjivan SF Bank  12 months 1 day  to 560 days(80 weeks)    8.00%   
8   Ujjivan SF Bank                    561 days to 990 days    7.75%   
9   Ujjivan SF Bank                 Tax-saver Fixed Deposit    7.20%   
10  Ujjivan SF Bank                   991 days to 60 months    7.20%   
11  Ujjivan SF Bank           60 mo